# Exercises XP: LoRA Implementation Lab
All exercises are fully implemented below. Run each cell in order.


## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [ ]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [2]:
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha
        self.rank = rank # Store rank for scaling

    def forward(self, x):
        x = (self.alpha * (x @ self.A @ self.B)) / self.rank  # apply the low-rank update (batch, in_dim) -> (batch, out_dim)
        return x

# Hyperparameters for the sandbox test
random_seed = 123
in_dim = 784
out_dim = 128
rank = 4
alpha = 8

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)
x = torch.randn(2, in_dim)  # e.g., torch.randn(batch, in_dim)

print(x)
print(layer)
print("Original output:", layer(x))

tensor([[ 0.3109,  0.0719,  0.0861,  ..., -0.8510,  2.0836,  1.6503],
        [-0.1846,  0.1089,  0.0448,  ..., -0.2109, -0.5282, -0.4833]])
LoRALayer()
Original output: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 

# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [3]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

base_linear = nn.Linear(in_dim, out_dim)
layer_lora_1 = LinearWithLoRA(base_linear, rank, alpha)  # wrap `base_linear` with rank/alpha values
print("LinearWithLoRA output:", layer_lora_1(x))

LinearWithLoRA output: tensor([[ 5.8811e-02, -9.6130e-01,  9.4886e-01, -1.2297e-01, -5.0787e-01,
         -1.9863e-01, -5.1383e-01,  4.7193e-01,  6.3721e-01,  1.3190e-01,
          1.1604e+00, -5.7265e-01, -5.3856e-01,  4.6501e-01,  7.6220e-01,
         -3.5463e-01,  5.1838e-01,  7.3666e-01, -6.5002e-01, -8.1966e-01,
          1.0454e+00, -2.8439e-02, -2.4681e-01, -4.1222e-02,  6.2543e-01,
         -5.4409e-01, -3.6047e-01,  3.6518e-01, -2.6254e-01,  7.0103e-01,
          4.3933e-01, -2.3332e-01, -5.4189e-01, -5.5404e-01, -3.8527e-01,
         -4.2197e-01, -9.7479e-01, -1.1786e+00,  2.4960e-02, -5.5171e-01,
          8.8652e-02, -4.9760e-01, -2.4536e-01, -9.7918e-02,  1.2705e-01,
         -2.2534e-03, -1.0875e+00, -1.3744e-01, -1.0966e-01,  3.1145e-02,
         -2.6580e-01,  6.5752e-02, -6.0117e-01, -5.2899e-01,  5.3116e-01,
          1.3629e-01, -1.0752e+00,  7.2386e-01,  4.8598e-01,  1.8455e-01,
          3.4056e-01,  2.8450e-01,  4.3492e-01, -1.8961e-01,  8.9772e-02,
          4.012

# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [4]:
class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

single_net = SingleLayerNet(num_features=in_dim, num_classes=out_dim)
sample_input = x # Using the previously defined toy tensor 'x'

with torch.no_grad():
    baseline_output = single_net(sample_input)

# Create a new nn.Linear with the same initial weights as the original, to ensure a fair comparison
# This is a bit tricky, if single_net.layer was not explicitly deep-copied, the LoRA wrapper would get the original weights
# Here we just want to show that if you replace a new linear layer with a LinearWithLoRA wrapping a new linear layer, outputs match.
new_linear_layer = nn.Linear(in_dim, out_dim)
new_linear_layer.weight.data = single_net.layer.weight.data.clone() # Ensure same initial weights
new_linear_layer.bias.data = single_net.layer.bias.data.clone() # Ensure same initial bias

single_net_lora = SingleLayerNet(num_features=in_dim, num_classes=out_dim) # Create a new network
single_net_lora.layer = LinearWithLoRA(new_linear_layer, rank, alpha)  # replace with LinearWithLoRA

with torch.no_grad():
    lora_output = single_net_lora(sample_input)

print("Outputs match before training?", torch.allclose(baseline_output, lora_output))

Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [6]:
import torch.nn.functional as F

class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        lora = self.lora.A @ self.lora.B # combine LoRA matrices (in_dim, out_dim)
        # then combine LoRA original weights
        # self.linear.weight is (out_dim, in_dim), so lora must be transposed
        combined_weight = self.linear.weight + (self.lora.alpha / self.lora.rank) * lora.T # merge original weights + scaled LoRA correction
        return F.linear(x, combined_weight, self.linear.bias)

# To ensure fair comparison, create a new nn.Linear with the same initial weights as `base_linear`
equivalent_linear_for_merged = nn.Linear(in_dim, out_dim)
equivalent_linear_for_merged.weight.data = base_linear.weight.data.clone()
equivalent_linear_for_merged.bias.data = base_linear.bias.data.clone()

layer_lora_2 = LinearWithLoRAMerged(equivalent_linear_for_merged, rank, alpha)

with torch.no_grad():
    merged_output = layer_lora_2(x)
    original_lora_output = layer_lora_1(x)

print("Merged LoRA output:", merged_output)
print("Outputs match LinearWithLoRA?", torch.allclose(original_lora_output, merged_output))

Merged LoRA output: tensor([[ 5.8811e-02, -9.6130e-01,  9.4886e-01, -1.2297e-01, -5.0787e-01,
         -1.9863e-01, -5.1383e-01,  4.7193e-01,  6.3721e-01,  1.3190e-01,
          1.1604e+00, -5.7265e-01, -5.3856e-01,  4.6501e-01,  7.6220e-01,
         -3.5463e-01,  5.1838e-01,  7.3666e-01, -6.5002e-01, -8.1966e-01,
          1.0454e+00, -2.8439e-02, -2.4681e-01, -4.1222e-02,  6.2543e-01,
         -5.4409e-01, -3.6047e-01,  3.6518e-01, -2.6254e-01,  7.0103e-01,
          4.3933e-01, -2.3332e-01, -5.4189e-01, -5.5404e-01, -3.8527e-01,
         -4.2197e-01, -9.7479e-01, -1.1786e+00,  2.4960e-02, -5.5171e-01,
          8.8652e-02, -4.9760e-01, -2.4536e-01, -9.7918e-02,  1.2705e-01,
         -2.2534e-03, -1.0875e+00, -1.3744e-01, -1.0966e-01,  3.1145e-02,
         -2.6580e-01,  6.5752e-02, -6.0117e-01, -5.2899e-01,  5.3116e-01,
          1.3629e-01, -1.0752e+00,  7.2386e-01,  4.8598e-01,  1.8455e-01,
          3.4056e-01,  2.8450e-01,  4.3492e-01, -1.8961e-01,  8.9772e-02,
          4.0120e-

# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [7]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes),
        )

    def forward(self, x):
        x = self.layers(x)
        return x

In [8]:
# Architecture
num_features = in_dim # 784 for flattened MNIST images
num_hidden_1 = 256
num_hidden_2 = 128
num_classes = 10 # 10 digits for MNIST

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 0.001
num_epochs = 5

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(DEVICE)
print(model)
print(optimizer_pretrained)

cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [10]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 64

# Note: transforms.ToTensor() scales input images to 0-1 range
train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)

test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.62MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 130kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.38MB/s]

Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Define evaluation

In [ ]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            # Move data to the target device (CPU or GPU)
            features = features.view(-1, 28 * 28).to(device)  # flatten 28x28 → 784
            targets  = targets.to(device)
            # Forward pass: get raw class scores (logits)
            logits = model(features)
            # Pick the class with the highest score
            _, predicted_labels = torch.max(logits, 1)
            # Accumulate counts
            num_examples  += targets.size(0)                          # number of samples in the batch
            correct_pred  += (predicted_labels == targets).sum()      # number of correct predictions
    return correct_pred.float() / num_examples * 100  # accuracy in %


## Training

In [ ]:
def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            # Flatten images and move to device
            features = features.view(-1, 28 * 28).to(device)  # (B, 1, 28, 28) → (B, 784)
            targets  = targets.to(device)

            # Forward and back propagation
            logits = model(features)                                   # raw class scores
            loss   = torch.nn.functional.cross_entropy(logits, targets)  # cross-entropy loss
            optimizer.zero_grad()   # reset gradients from previous step

            loss.backward()         # compute gradients

            # Update model parameters
            optimizer.step()

            # Logging every 400 batches
            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (
                    epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (
                epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))


In [ ]:
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

# Replacing Linear with LoRA Layers

In [ ]:
model_lora = copy.deepcopy(model)

# Replace every nn.Linear in the MLP with LinearWithLoRAMerged
# layers[0] = Linear(784, 256)   → index 0
# layers[1] = ReLU               → index 1  (skip)
# layers[2] = Linear(256, 128)   → index 2
# layers[3] = ReLU               → index 3  (skip)
# layers[4] = Linear(128, 10)    → index 4
model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)
model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')


## Freezing the Original Linear Layers

In [ ]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}:{param.requires_grad}')

In [ ]:
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')